# Chronos-2 Embeddings Demo

This notebook downloads a Chronos-2 model from Hugging Face and extracts embeddings for a few local time series.

In [1]:
import os
import sys
import inspect
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from chronos import BaseChronosPipeline

if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.append("..")

from tscglue.data_loader import load_fold

2026-02-25 13:04:12.513792: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
DATASET_NAME = "FordA"
FOLD = 0
X_train, y_train, X_test, y_test = load_fold(DATASET_NAME, fold=FOLD)

In [3]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tscglue.models_tsfm import Chronos2Classifier

m = Chronos2Classifier().fit(X_train, y_train)

Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[T

In [4]:
X_train_emb = m.compute_chronos2_embeddings_local(X_train)
X_test_emb = m.compute_chronos2_embeddings_local(X_test)

print("Train embeddings shape:", X_train_emb.shape)
print("Test embeddings shape:", X_test_emb.shape)


Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[T

In [5]:
P = m.predict_proba(X_test)
P

Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])


array([[0.9411508 , 0.0588492 ],
       [0.91484684, 0.08515316],
       [0.678993  , 0.321007  ],
       ...,
       [0.12930286, 0.87069714],
       [0.33754584, 0.66245416],
       [0.25815247, 0.74184753]], shape=(1320, 2))

In [6]:
y_pred_m = m.predict(X_test)
m_acc = accuracy_score(y_test, y_pred_m)

Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])
Succeeded with: embed(list[Tensor])


In [7]:
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_emb, y_train)
y_pred = rf.predict(X_test_emb)
rf_acc = accuracy_score(y_test, y_pred)
print(f"RF test accuracy on Chronos-2 embeddings: {rf_acc:.4f}")


RF test accuracy on Chronos-2 embeddings: 0.8492


In [8]:
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rf_pca = make_pipeline(
    PCA(n_components=0.90, svd_solver="full"),
    RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
)
rf_pca.fit(X_train_emb, y_train)
y_pred_rf_pca = rf_pca.predict(X_test_emb)
rf_pca_acc = accuracy_score(y_test, y_pred_rf_pca)
print(f"RF + PCA(90%) test accuracy: {rf_pca_acc:.4f}")

ridge = make_pipeline(
    StandardScaler(),
    RidgeClassifierCV(alphas=np.logspace(-3, 3, 13)),
)
ridge.fit(X_train_emb, y_train)
y_pred_ridge = ridge.predict(X_test_emb)
ridge_acc = accuracy_score(y_test, y_pred_ridge)
print(f"RidgeClassifierCV test accuracy: {ridge_acc:.4f}")


RF + PCA(90%) test accuracy: 0.8652
RidgeClassifierCV test accuracy: 0.9341


In [9]:
from aeon.classification.convolution_based import MultiRocketHydraClassifier

mrh = MultiRocketHydraClassifier(random_state=42, n_jobs=-1)
mrh.fit(X_train.astype("float64"), y_train)
y_pred_mrh = mrh.predict(X_test.astype("float64"))
mrh_acc = accuracy_score(y_test, y_pred_mrh)

print(f"MultiRocketHydra test accuracy: {mrh_acc:.4f}")


MultiRocketHydra test accuracy: 0.9561


In [10]:
comparison_df = pd.DataFrame(
    [
        {"model": "RF on Chronos-2 [REG] token", "test_accuracy": rf_acc},
        {"model": "RF + PCA(90%) on Chronos-2 [REG] token", "test_accuracy": rf_pca_acc},
        {"model": "RidgeClassifierCV on Chronos-2 [REG] token", "test_accuracy": ridge_acc},
        {"model": "MultiRocketHydraClassifier", "test_accuracy": mrh_acc},
        {"model": "mine", "test_accuracy": m_acc}
    ]
).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
comparison_df


,model,test_accuracy
0,MultiRocketHydraClassifier,0.956061
1,mine,0.934091
2,RidgeClassifierCV on Chronos-2 [REG] token,0.934091
3,RF + PCA(90%) on Chronos-2 [REG] token,0.865152
4,RF on Chronos-2 [REG] token,0.849242
